In [1]:
# %%
from pathlib import Path
import json
import time
import requests
import pandas as pd
from tqdm import tqdm
from lxml import etree

import boto3
from botocore import UNSIGNED
from botocore.client import Config

In [2]:
# %%
PROJECT_ROOT = Path("..").resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PARSED_DIR = PROJECT_ROOT / "data" / "parsed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PARSED_DIR.mkdir(parents=True, exist_ok=True)

CONDITION = "sepsis"
MAX_ARTICLES = 1000
REQUEST_SLEEP = 0.2

In [3]:
# %%
def pmc_esearch(term: str, retmax: int) -> list[str]:
    query = (
        f"{term} AND "
        "(cc0_license[filter] OR "
        "cc_by_license[filter] OR "
        "cc_by-sa_license[filter] OR "
        "cc_by-nd_license[filter])"
    )

    r = requests.get(
        "https://eutils.ncbi.nlm.nih.gov/eutils/esearch.fcgi",
        params={
            "db": "pmc",
            "term": query,
            "retmax": retmax,
            "format": "json",
        },
        timeout=30,
    )
    r.raise_for_status()
    return r.json()["esearchresult"]["idlist"]

In [4]:
# %%
pmc_ids = pmc_esearch(CONDITION, MAX_ARTICLES)
print(f"PMC IDs found: {len(pmc_ids)}")

PMC IDs found: 1000


In [5]:
# %%
s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED),
)

In [6]:
# %%
def list_versions(pmc_id: str) -> list[str]:
    prefix = f"PMC{pmc_id}."
    resp = s3.list_objects_v2(
        Bucket="pmc-oa-opendata",
        Prefix=prefix,
        Delimiter="/",
    )
    return [p["Prefix"] for p in resp.get("CommonPrefixes", [])]

In [7]:
# %%
def download_version(version_prefix: str):
    out_dir = RAW_DIR / version_prefix.rstrip("/")
    out_dir.mkdir(exist_ok=True)

    base = version_prefix.rstrip("/")
    for ext in ("xml", "json"):
        key = f"{version_prefix}{base}.{ext}"
        try:
            s3.download_file(
                "pmc-oa-opendata",
                key,
                str(out_dir / f"{base}.{ext}"),
            )
        except Exception:
            pass

In [8]:
# %%
downloaded = 0

for pmc_id in tqdm(pmc_ids):
    for v in list_versions(pmc_id):
        download_version(v)
        downloaded += 1
    time.sleep(REQUEST_SLEEP)

print(f"Downloaded article versions: {downloaded}")

100%|██████████| 1000/1000 [10:43<00:00,  1.55it/s]

100%|██████████| 1000/1000 [10:43<00:00,  1.55it/s]

Downloaded article versions: 1003


In [9]:
# %%
def parse_xml(xml_path: Path):
    parser = etree.XMLParser(recover=True)
    tree = etree.parse(str(xml_path), parser)
    root = tree.getroot()

    def xp(expr):
        return root.xpath(expr)

    # Article title
    title_nodes = xp(".//*[local-name()='article-title']")
    article_title = (
        "".join(title_nodes[0].itertext()).strip()
        if title_nodes and "".join(title_nodes[0].itertext()).strip()
        else None
    )

    records = []

    for sec in xp(".//*[local-name()='sec']"):
        title_el = sec.xpath(".//*[local-name()='title']")
        section_title = (
            "".join(title_el[0].itertext()).strip()
            if title_el and "".join(title_el[0].itertext()).strip()
            else "UNKNOWN"
        )

        paragraphs = sec.xpath(".//*[local-name()='p']")
        for i, p in enumerate(paragraphs):
            text = " ".join(p.itertext()).strip()
            if len(text) >= 50:
                records.append((section_title, i, text))

    return article_title, records

In [10]:
# %%
rows = []

for article_dir in tqdm(RAW_DIR.iterdir()):
    xmls = list(article_dir.glob("*.xml"))
    jsons = list(article_dir.glob("*.json"))

    if not xmls or not jsons:
        continue

    meta = json.loads(jsons[0].read_text())
    if meta.get("is_retracted", False):
        continue

    pmcid = meta["pmcid"]
    license_code = meta.get("license_code")

    try:
        version = int(article_dir.name.split(".")[-1])
    except Exception:
        continue

    article_title, paragraphs = parse_xml(xmls[0])

    for sec, idx, txt in paragraphs:
        rows.append({
            "pmcid": pmcid,
            "version": version,
            "article_title": article_title,
            "section_raw": sec,
            "paragraph_index": idx,
            "paragraph_text": txt,
            "license": license_code,
            "is_retracted": False,
        })

1003it [00:46, 21.42it/s]


In [11]:
# %%
df = pd.DataFrame(rows)

out_path = PARSED_DIR / "pmc_raw_articles.parquet"
df.to_parquet(out_path, index=False)

In [12]:
# %%
print("==== NOTEBOOK 01 AUDIT ====\n")

print("Rows:", len(df))

if df.empty:
    print("❌ DataFrame is empty — ingestion failed")
    print("Columns:", df.columns.tolist())
else:
    print("Unique PMCIDs:", df["pmcid"].nunique())

    print("\nVersions per PMC (top 5):")
    print(df.groupby("pmcid")["version"].nunique().sort_values(ascending=False).head())

    print("\nLicenses:")
    print(df["license"].value_counts())

    print("\nTop raw sections:")
    print(df["section_raw"].value_counts().head(10))

    print("\nParagraph length stats:")
    print(df["paragraph_text"].str.len().describe())

    print("\nSample rows:")
    display(df.sample(3, random_state=42))

print("\nOutput file:")
print(out_path.resolve())

==== NOTEBOOK 01 AUDIT ====

Rows: 88995
Unique PMCIDs: 994

Versions per PMC (top 5):
pmcid
PMC12446057    2
PMC12885954    2
PMC12880088    2
PMC7618728     1
PMC12919932    1
Name: version, dtype: int64

Licenses:
license
CC BY       88705
CC0           138
CC BY-ND       89
TDM            63
Name: count, dtype: int64

Top raw sections:
section_raw
Results                     9268
Discussion                  5215
Methods                     3542
Introduction                3018
Materials and methods       2496
3. Results                  1959
4. Discussion               1409
1. Introduction             1089
2. Materials and Methods    1007
Case presentation            706
Name: count, dtype: int64

Paragraph length stats:
count    88995.000000
mean       723.739817
std        681.580391
min         50.000000
25%        307.000000
50%        577.000000
75%        948.500000
max      45258.000000
Name: paragraph_text, dtype: float64

Sample rows:


,pmcid,version,article_title,section_raw,paragraph_index,paragraph_text,license,is_retracted
1040,PMC12894070,1,Isolated anal tuberculosis presenting as an an...,DISCUSSION,2,The clinical presentation of anal tuberculosis...,CC BY,False
19838,PMC12901514,1,Neoepitopes at the crossroads of immunometabol...,Neoepitope formation through oxidative and red...,3,Nitrosative stress contributes further to the ...,CC BY,False
29744,PMC12907296,1,Lipid metabolism in homeostasis and disease,Membrane lipids regulate immune cell responses,5,"Phosphoinositides are glycerophospholipids, wi...",CC BY,False



Output file:
D:\Pictures\pmc_graphrag\data\parsed\pmc_raw_articles.parquet


In [14]:
# =========================
# NOTEBOOK 01 — INGESTION METRICS REPORT
# =========================
from collections import Counter
import pandas as pd
import json

print("=" * 90)
print("NOTEBOOK 01 METRICS — DOWNLOAD + PARSE AUDIT")
print("=" * 90)

# --- Request / retrieval level ---
print(f"Condition query: {CONDITION}")
print(f"Requested MAX_ARTICLES: {MAX_ARTICLES}")
print(f"PMC IDs returned by eSearch: {len(pmc_ids) if 'pmc_ids' in globals() else 'N/A'}")
print(f"Downloaded article versions: {downloaded if 'downloaded' in globals() else 'N/A'}")

# --- Raw directory scan ---
article_dirs = [p for p in RAW_DIR.iterdir() if p.is_dir()]
n_article_dirs = len(article_dirs)

missing_xml_json = 0
retracted_count = 0
parse_fail_count = 0
version_counter = Counter()
license_counter = Counter()

for article_dir in article_dirs:
    xmls = list(article_dir.glob("*.xml"))
    jsons = list(article_dir.glob("*.json"))

    if not xmls or not jsons:
        missing_xml_json += 1
        continue

    try:
        meta = json.loads(jsons[0].read_text())
        if meta.get("is_retracted", False):
            retracted_count += 1
        if meta.get("license_code"):
            license_counter[meta.get("license_code")] += 1
    except Exception:
        pass

    try:
        version = int(article_dir.name.split(".")[-1])
        version_counter[version] += 1
    except Exception:
        pass

    try:
        _ = parse_xml(xmls[0])
    except Exception:
        parse_fail_count += 1

print("\n[Filesystem audit]")
print(f"Raw article directories present: {n_article_dirs}")
print(f"Directories missing XML or JSON: {missing_xml_json}")
print(f"Retracted articles detected in metadata: {retracted_count}")
print(f"XML parse failures (re-scan): {parse_fail_count}")

# --- DataFrame-level metrics ---
if 'df' not in globals() or df.empty:
    print("\nNo parsed DataFrame available or DataFrame is empty.")
else:
    print("\n[Parsed parquet audit]")
    print(f"Parsed rows: {len(df):,}")
    print(f"Unique PMCIDs: {df['pmcid'].nunique():,}")
    print(f"Unique article titles: {df['article_title'].nunique():,}")
    print(f"Unique licenses: {df['license'].nunique():,}")

    pmcid_version_counts = df.groupby("pmcid")["version"].nunique()
    rows_per_article = df.groupby("pmcid").size()
    paras_per_section = df.groupby(["pmcid", "section_raw"]).size()

    print("\nVersions per PMCID:")
    display(pmcid_version_counts.describe())

    print("\nRows per article:")
    display(rows_per_article.describe())

    print("\nParagraphs per (PMCID, raw section):")
    display(paras_per_section.describe())

    print("\nLicense distribution:")
    display(df["license"].value_counts(dropna=False).to_frame("count"))

    print("\nTop raw sections:")
    display(df["section_raw"].value_counts().head(15).to_frame("count"))

    para_len = df["paragraph_text"].str.len()
    print("\nParagraph length stats:")
    display(para_len.describe())

    print("\nApprox parse success rate vs eSearch IDs:")
    if 'pmc_ids' in globals() and len(pmc_ids) > 0:
        parse_success_rate = df["pmcid"].nunique() / len(pmc_ids)
        print(f"{parse_success_rate:.2%}")
    else:
        print("N/A")

    print("\nVersion distribution from raw dirs:")
    if version_counter:
        display(pd.Series(version_counter).sort_index().to_frame("count"))
    else:
        print("No version data found.")

    print("\nLicense distribution from raw metadata scan:")
    if license_counter:
        display(pd.Series(license_counter).sort_values(ascending=False).to_frame("count"))
    else:
        print("No license metadata found.")

print("\nOutput parquet:")
print(out_path.resolve() if 'out_path' in globals() else "N/A")

NOTEBOOK 01 METRICS — DOWNLOAD + PARSE AUDIT
Condition query: sepsis
Requested MAX_ARTICLES: 1000
PMC IDs returned by eSearch: 1000
Downloaded article versions: 1003

[Filesystem audit]
Raw article directories present: 1003
Directories missing XML or JSON: 0
Retracted articles detected in metadata: 0
XML parse failures (re-scan): 0

[Parsed parquet audit]
Parsed rows: 88,995
Unique PMCIDs: 994
Unique article titles: 994
Unique licenses: 4

Versions per PMCID:


count    994.000000
mean       1.003018
std        0.054882
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        2.000000
Name: version, dtype: float64


Rows per article:


count    994.000000
mean      89.532193
std       70.112408
min        1.000000
25%       47.000000
50%       76.000000
75%      111.500000
max      775.000000
dtype: float64


Paragraphs per (PMCID, raw section):


...


Approx parse success rate vs eSearch IDs:
99.40%

Version distribution from raw dirs:


,count
1,1000
2,3



License distribution from raw metadata scan:


,count
CC BY,1000
TDM,1
CC BY-ND,1
CC0,1



Output parquet:
D:\Pictures\pmc_graphrag\data\parsed\pmc_raw_articles.parquet
